# ViT LoRA 训练探索笔记

本笔记演示如何使用LoRA技术对Vision Transformer进行高效微调。

## 目录
1. [环境设置](#环境设置)
2. [数据加载和可视化](#数据加载和可视化)
3. [模型创建和LoRA配置](#模型创建和LoRA配置)
4. [训练设置和监控](#训练设置和监控)
5. [模型评估](#模型评估)
6. [特征提取和分析](#特征提取和分析)

## 1. 环境设置

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import yaml
from tqdm import tqdm

# 添加项目路径
project_root = os.path.dirname(os.getcwd())
sys.path.append(project_root)

# 导入项目模块
from src.models import create_vit_lora_model, print_trainable_parameters, get_lora_config
from src.data import get_train_transforms, get_val_transforms
from src.training import set_seed, get_device

# 设置随机种子
set_seed(42)

# 获取设备
device = get_device()
print(f"使用设备: {device}")

## 2. 数据加载和可视化

In [ ]:
# 使用CIFAR-10作为示例数据集
# CIFAR-10类别名称
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# 定义数据变换
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 加载CIFAR-10数据集
train_dataset = torchvision.datasets.CIFAR10(
    root='../data/raw', train=True, download=True, transform=train_transform
)

val_dataset = torchvision.datasets.CIFAR10(
    root='../data/raw', train=False, download=True, transform=val_transform
)

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(val_dataset)}")
print(f"类别数: {len(CIFAR10_CLASSES)}")

In [ ]:
# 可视化数据样本
def show_samples(dataset, num_samples=16, title="数据样本"):
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    fig.suptitle(title, fontsize=16)
    
    for i in range(num_samples):
        img, label = dataset[i]
        
        # 反标准化
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img * std + mean
        img = torch.clamp(img, 0, 1)
        
        # 转换为PIL图像
        img_pil = transforms.ToPILImage()(img)
        
        row, col = i // 4, i % 4
        axes[row, col].imshow(img_pil)
        axes[row, col].set_title(f"{CIFAR10_CLASSES[label]}")
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

# 显示训练样本
show_samples(train_dataset, title="CIFAR-10 训练样本")

## 3. 模型创建和LoRA配置

In [ ]:
# 创建ViT LoRA模型
model = create_vit_lora_model(
    model_name="google/vit-base-patch16-224",
    num_classes=10,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"]
)

# 打印模型信息
print_trainable_parameters(model)

# 将模型移动到设备
model = model.to(device)

In [ ]:
# 查看不同LoRA配置的参数数量对比
configs = {
    'base': {'r': 8, 'alpha': 16},
    'medium': {'r': 16, 'alpha': 32},
    'large': {'r': 32, 'alpha': 64}
}

print("不同LoRA配置的参数对比:")
for name, config in configs.items():
    temp_model = create_vit_lora_model(
        model_name="google/vit-base-patch16-224",
        num_classes=10,
        lora_r=config['r'],
        lora_alpha=config['alpha'],
        target_modules=["query", "key", "value", "dense"]
    )
    print(f"\n{name.upper()} 配置 (r={config['r']}, alpha={config['alpha']}):")
    print_trainable_parameters(temp_model)
    del temp_model  # 释放内存

## 4. 训练设置和监控

In [ ]:
# 训练配置
training_config = {
    'num_epochs': 5,  # 为了演示，使用较少的epoch
    'learning_rate': 1e-4,
    'weight_decay': 1e-2,
    'use_amp': True,  # 混合精度训练
    'checkpoint_dir': '../checkpoints',
    'log_dir': '../logs'
}

# 创建优化器和损失函数
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=training_config['learning_rate'],
    weight_decay=training_config['weight_decay']
)

criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=training_config['num_epochs']
)

# 混合精度训练器
if training_config['use_amp']:
    scaler = torch.cuda.amp.GradScaler()

print("训练配置:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

In [ ]:
# 训练函数
def train_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(loader, desc='训练中')
    
    for batch_idx, (images, labels) in enumerate(progress_bar):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs.logits, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # 更新进度条
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{100. * correct / total:.2f}%'
        })
        
        # 为了演示，只训练一部分数据
        if batch_idx >= 50:  # 限制训练步数
            break
    
    avg_loss = total_loss / min(len(loader), 50)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy

In [ ]:
# 验证函数
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        progress_bar = tqdm(loader, desc='验证中')
        
        for batch_idx, (images, labels) in enumerate(progress_bar):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs.logits, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100. * correct / total:.2f}%'
            })
            
            # 为了演示，只验证一部分数据
            if batch_idx >= 20:
                break
    
    avg_loss = total_loss / min(len(loader), 20)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy

In [ ]:
# 开始训练
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print("开始训练...")
for epoch in range(training_config['num_epochs']):
    print(f"\nEpoch {epoch + 1}/{training_config['num_epochs']}")
    print("-" * 30)
    
    # 训练
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, criterion, 
        scaler if training_config['use_amp'] else None
    )
    
    # 验证
    val_loss, val_acc = validate_epoch(model, val_loader, criterion)
    
    # 更新学习率
    scheduler.step()
    
    # 记录结果
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    print(f"训练损失: {train_loss:.4f}, 训练准确率: {train_acc:.2f}%")
    print(f"验证损失: {val_loss:.4f}, 验证准确率: {val_acc:.2f}%")
    print(f"学习率: {optimizer.param_groups[0]['lr']:.6f}")

print("\n训练完成！")

## 5. 模型评估

In [ ]:
# 绘制训练曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 损失曲线
ax1.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='训练损失')
ax1.plot(range(1, len(val_losses) + 1), val_losses, 'r-', label='验证损失')
ax1.set_title('训练和验证损失')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('损失')
ax1.legend()
ax1.grid(True)

# 准确率曲线
ax2.plot(range(1, len(train_accs) + 1), train_accs, 'b-', label='训练准确率')
ax2.plot(range(1, len(val_accs) + 1), val_accs, 'r-', label='验证准确率')
ax2.set_title('训练和验证准确率')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('准确率 (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 详细评估
def detailed_evaluation(model, loader, class_names, num_batches=10):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(loader):
            if batch_idx >= num_batches:
                break
                
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(outputs.logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # 计算每类准确率
    class_accuracy = {}
    for i, class_name in enumerate(class_names):
        class_mask = all_labels == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(all_preds[class_mask] == all_labels[class_mask])
            class_accuracy[class_name] = class_acc * 100
    
    return all_preds, all_labels, all_probs, class_accuracy

# 进行详细评估
preds, labels, probs, class_acc = detailed_evaluation(model, val_loader, CIFAR10_CLASSES)

print("各类别准确率:")
for class_name, acc in class_acc.items():
    print(f"  {class_name}: {acc:.2f}%")

overall_acc = np.mean(preds == labels) * 100
print(f"\n总体准确率: {overall_acc:.2f}%")

In [ ]:
# 可视化预测结果
def visualize_predictions(model, dataset, class_names, num_samples=12):
    model.eval()
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle('预测结果示例', fontsize=16)
    
    with torch.no_grad():
        for i in range(num_samples):
            # 获取样本
            img, true_label = dataset[i]
            img_batch = img.unsqueeze(0).to(device)
            
            # 预测
            outputs = model(img_batch)
            probs = torch.softmax(outputs.logits, dim=1)
            pred_label = torch.argmax(outputs.logits, dim=1).item()
            confidence = probs[0, pred_label].item()
            
            # 反标准化图像
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            img_denorm = img * std + mean
            img_denorm = torch.clamp(img_denorm, 0, 1)
            
            # 显示图像
            row, col = i // 4, i % 4
            axes[row, col].imshow(transforms.ToPILImage()(img_denorm))
            
            # 设置标题
            true_name = class_names[true_label]
            pred_name = class_names[pred_label]
            is_correct = pred_label == true_label
            
            title = f"真实: {true_name}\n预测: {pred_name}\n置信度: {confidence:.3f}"
            color = 'green' if is_correct else 'red'
            axes[row, col].set_title(title, color=color, fontsize=10)
            axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

# 显示预测结果
visualize_predictions(model, val_dataset, CIFAR10_CLASSES)

## 6. 特征提取和分析

In [ ]:
# 特征提取器
class FeatureExtractor:
    def __init__(self, model):
        self.model = model
        self.features = {}
        self.hooks = []
        
    def register_hooks(self):
        def hook_fn(name):
            def hook(module, input, output):
                self.features[name] = output.detach()
            return hook
        
        # 为不同层注册hooks
        if hasattr(self.model, 'vit'):
            # ViT encoder的最后一层
            self.hooks.append(
                self.model.vit.encoder.layer[-1].register_forward_hook(
                    hook_fn('last_encoder_layer')
                )
            )
            # ViT的池化层
            self.hooks.append(
                self.model.vit.pooler.register_forward_hook(
                    hook_fn('pooler')
                )
            )
    
    def extract_features(self, x):
        self.features.clear()
        with torch.no_grad():
            _ = self.model(x)
        return self.features
    
    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()

# 创建特征提取器
feature_extractor = FeatureExtractor(model)
feature_extractor.register_hooks()

print("特征提取器已创建")

In [ ]:
# 提取特征并进行可视化
def extract_and_visualize_features(extractor, dataset, num_samples=100):
    model.eval()
    
    features_list = []
    labels_list = []
    
    print("提取特征...")
    for i in tqdm(range(num_samples)):
        img, label = dataset[i]
        img_batch = img.unsqueeze(0).to(device)
        
        features = extractor.extract_features(img_batch)
        
        # 获取池化后的特征
        if 'pooler' in features:
            pooled_features = features['pooler'].squeeze().cpu().numpy()
            features_list.append(pooled_features)
            labels_list.append(label)
    
    return np.array(features_list), np.array(labels_list)

# 提取特征
features, feature_labels = extract_and_visualize_features(feature_extractor, val_dataset)
print(f"特征形状: {features.shape}")
print(f"标签形状: {feature_labels.shape}")

In [ ]:
# 使用t-SNE进行特征可视化
from sklearn.manifold import TSNE
import matplotlib.colors as mcolors

# 使用t-SNE降维
print("进行t-SNE降维...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features)

# 可视化特征分布
plt.figure(figsize=(12, 10))
colors = plt.cm.tab10(np.linspace(0, 1, len(CIFAR10_CLASSES)))

for i, class_name in enumerate(CIFAR10_CLASSES):
    mask = feature_labels == i
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1], 
               c=[colors[i]], label=class_name, alpha=0.7, s=20)

plt.title('ViT特征的t-SNE可视化', fontsize=16)
plt.xlabel('t-SNE 维度 1')
plt.ylabel('t-SNE 维度 2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("特征可视化完成！")

In [ ]:
# 分析LoRA层的权重
def analyze_lora_weights(model):
    print("LoRA层权重分析:")
    
    lora_weights = {}
    
    for name, param in model.named_parameters():
        if 'lora' in name.lower():
            weight_stats = {
                'mean': param.data.mean().item(),
                'std': param.data.std().item(),
                'min': param.data.min().item(),
                'max': param.data.max().item(),
                'norm': param.data.norm().item()
            }
            lora_weights[name] = weight_stats
            
            print(f"\n{name}:")
            print(f"  形状: {param.shape}")
            print(f"  均值: {weight_stats['mean']:.6f}")
            print(f"  标准差: {weight_stats['std']:.6f}")
            print(f"  范围: [{weight_stats['min']:.6f}, {weight_stats['max']:.6f}]")
            print(f"  L2范数: {weight_stats['norm']:.6f}")
    
    return lora_weights

# 分析LoRA权重
lora_analysis = analyze_lora_weights(model)

In [ ]:
# 清理
feature_extractor.remove_hooks()
print("特征提取器hooks已清理")

# 保存模型
torch.save({
    'model_state_dict': model.state_dict(),
    'train_losses': train_losses,
    'train_accs': train_accs,
    'val_losses': val_losses,
    'val_accs': val_accs,
    'class_names': CIFAR10_CLASSES
}, '../checkpoints/vit_lora_demo.pth')

print("模型已保存到 ../checkpoints/vit_lora_demo.pth")

## 总结

本笔记演示了完整的ViT LoRA训练流程：

1. **数据准备**: 使用CIFAR-10数据集，包括数据加载、预处理和可视化
2. **模型创建**: 创建ViT模型并应用LoRA技术，显著减少了可训练参数
3. **训练过程**: 实现了完整的训练循环，包括混合精度训练和学习率调度
4. **模型评估**: 评估模型性能，包括整体准确率和各类别准确率
5. **特征分析**: 使用t-SNE可视化学到的特征表示
6. **权重分析**: 分析LoRA层的权重分布

### 关键观察:
- LoRA技术大幅减少了可训练参数（通常减少90%以上）
- 训练效率显著提高，同时保持了良好的性能
- 特征表示显示了良好的类别分离性
- LoRA权重收敛到合理的数值范围

### 下一步:
- 尝试不同的LoRA配置（rank, alpha等）
- 测试不同的目标模块组合
- 在更大的数据集上进行实验
- 探索其他微调技术的组合